# Tutorial: Decoherence with Synthetic Noise Data

This notebook uses only public-safe synthetic inputs. It keeps the compact CSV spectrum for the electronics-chain example, and then switches to a broadband synthetic 1/f spectrum for the Ramsey / Spin Echo dephasing plot so the low-frequency physics and units stay self-consistent.


## Outline

1. Inspect the compact synthetic PSD window shipped with the public demo.
2. Build the `ElectronicNoise` pipeline and read out white-noise diagnostics.
3. Build a broadband synthetic 1/f PSD for dephasing, then run Ramsey / Spin Echo with the correct sensitivity scaling.
4. Compare how attenuation changes the white-noise temperature seen by the control line.


In [ ]:
from __future__ import annotations

import importlib.util
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

required = ["numpy", "matplotlib", "qutip"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        "Install the runtime dependencies first: " + ", ".join(missing)
    )

from pysuqu.decoherence import ElectronicNoise, ZNoiseDecoherence

repo_root = Path.cwd()
if repo_root.name == "demo":
    repo_root = repo_root.parent

window_data_path = repo_root / "demo" / "data" / "synthetic_flux_noise_psd.csv"
psd_table = np.genfromtxt(window_data_path, delimiter=",", names=True)
window_freq = psd_table["frequency_hz"]
window_psd_single = psd_table["psd_single_a2_per_hz"]

{
    "window_freq_range_hz": [float(window_freq[0]), float(window_freq[-1])],
    "window_point_count": int(window_freq.size),
}


## Step 1 - Inspect the compact synthetic PSD window

The CSV bundled in `demo/data/` is a small, public-safe measurement-style window that starts at `1e6 Hz`. That makes it a good fit for demonstrating the electronics-noise chain, but not for a full Ramsey dephasing example, because 1/f dephasing is dominated by much lower frequencies.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(window_freq, window_psd_single, marker="o")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Single-sided PSD (A^2/Hz)")
ax.set_title("Compact synthetic PSD window used for electronics-chain examples")
ax.grid(alpha=0.3, which="both")
plt.show()

{
    "window_low_freq_hz": float(window_freq[0]),
    "window_high_freq_hz": float(window_freq[-1]),
    "window_psd_first": float(window_psd_single[0]),
    "window_psd_last": float(window_psd_single[-1]),
}


## Step 2 - Build the electronic-noise pipeline

Here we keep using the compact CSV window and let `ElectronicNoise` propagate it through a representative attenuation chain. This is the right place to look at white-noise temperatures and fit diagnostics; the dephasing example in the next section will intentionally use a separate broadband synthetic 1/f spectrum.


In [ ]:
noise = ElectronicNoise(
    psd_freq=window_freq,
    psd_S=window_psd_single,
    noise_type="1f",
    noise_prop="single",
    is_spectral=True,
    T_setup=np.array([290, 45, 3.5, 0.9, 0.1, 0.01]),
    attenuation_setup=np.array([40, 1, 10, 10, 20, 10]),
    is_print=False,
)

{
    "input_white_noise_a2_per_hz": float(noise.input_stage.white_noise),
    "output_white_noise_a2_per_hz": float(noise.output_stage.white_noise),
    "input_white_noise_temperature_k": float(noise.input_stage.white_noise_temperature),
    "output_white_noise_temperature_k": float(noise.output_stage.white_noise_temperature),
    "corner_frequency_hz": float(noise.output_stage.fit_result.fit_diagnostics["corner_freq"]),
}


## Step 3 - Run a broadband synthetic dephasing workflow

This section fixes the scale issue in the original draft in two places:

- The coefficient passed to `cal_dephase(...)` must be `get_sensitivity_at_idle(...) * couple_term`, rather than the raw sensitivity alone.
- The synthetic dephasing PSD must include a low-frequency window, because Ramsey / Echo dephasing is driven by low-frequency 1/f noise. For this tutorial we use a public-safe broadband spectrum that starts at `2e4 Hz`, which also keeps `cal_tphi2(method='cal')` finite for the chosen delay window.


In [ ]:
dephasing_freq = np.logspace(np.log10(2e4), 8, 4000)
dephasing_psd_single = 5e-16 / dephasing_freq + 4e-21

with redirect_stdout(StringIO()):
    z_noise = ZNoiseDecoherence(
        psd_freq=dephasing_freq,
        psd_S=dephasing_psd_single,
        noise_type="1f",
        noise_prop="single",
        qubit_freq=5e9,
        qubit_anharm=-250e6,
        couple_term=1.5e-12,
        is_print=False,
    )

ramsey_idle_detuning_ghz = 0.1
ramsey_idle_freq = z_noise.qubit_freq - ramsey_idle_detuning_ghz * 1e9
sensitivity = z_noise.get_sensitivity_at_idle(idle_freq=ramsey_idle_freq)
sensitivity_factor = sensitivity * z_noise.couple_term

delay_list = np.linspace(0.05, 50.0, 120) * 1e-6
ramsey_prob = z_noise.cal_dephase(
    psd=dephasing_psd_single,
    noise_freq=dephasing_freq,
    sensitivity_factor=sensitivity_factor,
    experiment="Ramsey",
    delay_list=delay_list,
)
echo_prob = z_noise.cal_dephase(
    psd=dephasing_psd_single,
    noise_freq=dephasing_freq,
    sensitivity_factor=sensitivity_factor,
    experiment="SpinEcho",
    delay_list=delay_list,
)
tphi2_result = z_noise.cal_tphi2(
    method="cal",
    experiment="Ramsey",
    delay_list=delay_list,
    idle_freq=ramsey_idle_freq,
    is_print=False,
    is_plot=False,
)

fig, (ax_psd, ax_env) = plt.subplots(1, 2, figsize=(11, 4))
ax_psd.loglog(dephasing_freq, dephasing_psd_single, color="tab:purple")
ax_psd.set_xlabel("Frequency (Hz)")
ax_psd.set_ylabel("Synthetic PSD (A^2/Hz)")
ax_psd.set_title("Broadband synthetic 1/f PSD for dephasing")
ax_psd.grid(alpha=0.3, which="both")

ax_env.plot(delay_list * 1e6, ramsey_prob, label="Ramsey", linewidth=2)
ax_env.plot(delay_list * 1e6, echo_prob, label="Spin Echo", linewidth=2)
ax_env.set_xlabel("Delay (us)")
ax_env.set_ylabel("Coherence envelope")
ax_env.set_title("Dephasing envelopes with correct sensitivity scaling")
ax_env.grid(alpha=0.3)
ax_env.legend()

plt.tight_layout()
plt.show()

{
    "window_low_freq_hz": float(window_freq[0]),
    "dephasing_low_freq_hz": float(dephasing_freq[0]),
    "idle_freq_ghz": float(ramsey_idle_freq / 1e9),
    "sensitivity_rad_per_s_per_wb": float(sensitivity),
    "sensitivity_factor_rad_per_s_per_a": float(sensitivity_factor),
    "ramsey_start": float(ramsey_prob[0]),
    "ramsey_end": float(ramsey_prob[-1]),
    "echo_end": float(echo_prob[-1]),
    "analytic_tphi2_us": float(tphi2_result.value * 1e6),
}


## Exercises

Try changing the attenuation chain in the compact electronics example, or rescaling the broadband `1/f` coefficient in the dephasing cell to see how quickly the Ramsey envelope collapses relative to Spin Echo.


In [ ]:
def output_white_noise_temperature(attenuation_setup: np.ndarray) -> float:
    model = ElectronicNoise(
        psd_freq=window_freq,
        psd_S=window_psd_single,
        noise_type="1f",
        noise_prop="single",
        attenuation_setup=attenuation_setup,
        is_print=False,
    )
    return float(model.output_stage.white_noise_temperature)

{
    "default_chain_k": output_white_noise_temperature(np.array([40, 1, 10, 10, 20, 10])),
    "lighter_chain_k": output_white_noise_temperature(np.array([20, 1, 6, 6, 10, 6])),
    "heavier_chain_k": output_white_noise_temperature(np.array([50, 3, 12, 12, 20, 10])),
}
